# Texas Target Weights

This notebook scales Texas ZIP capacities to the deduplicated Olist customer count, so the target side matches the unique-customer source-code summary.


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "rebuild_unique_customer_mapping.py").exists():
            return candidate
    raise FileNotFoundError("Could not find rebuild_unique_customer_mapping.py from the current notebook path.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.options.display.max_columns = 120

from rebuild_unique_customer_mapping import (
    OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV,
    TEXAS_TARGET_WEIGHTS_CSV,
    build_texas_target_weights,
)


In [ ]:
unique_customers_df = pd.read_csv(OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV)
total_unique_customers = len(unique_customers_df)

print(f"Loaded unique Olist customers: {total_unique_customers:,}")
display(unique_customers_df.head())


In [ ]:
texas_target_weights_df = build_texas_target_weights(total_unique_customers)

print(f"Texas ZIP rows with positive population and density: {len(texas_target_weights_df):,}")
print(f"Target ZIP capacity total: {texas_target_weights_df['target_zip_capacity'].sum():,.6f}")
display(texas_target_weights_df.head(20))


In [ ]:
texas_target_weights_df.to_csv(TEXAS_TARGET_WEIGHTS_CSV, index=False)

print(f"Wrote Texas target weights: {TEXAS_TARGET_WEIGHTS_CSV}")


In [ ]:
validation = pd.DataFrame(
    {
        "check": [
            "target ZIP proportion total",
            "target ZIP capacity total",
            "unique customer total",
            "missing target capacities",
            "nonpositive target capacities",
        ],
        "value": [
            texas_target_weights_df["target_zip_proportion"].sum(),
            texas_target_weights_df["target_zip_capacity"].sum(),
            total_unique_customers,
            texas_target_weights_df["target_zip_capacity"].isna().sum(),
            (texas_target_weights_df["target_zip_capacity"] <= 0).sum(),
        ],
    }
)

display(validation)
